# 03 转化漏斗 + 品类拆解

回答：**用户怎么买东西？哪里卡住了？不同品类差异多大？**

In [ ]:
import sys; sys.path.append('..')
from utils.db import query
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
df = query("SELECT * FROM user_behavior_clean WHERE date < '2017-12-08'")

## 1. 整体转化漏斗

关键定义：每一步的**分母是上一步到达的人数**，不是总 UV。这决定了漏斗的正确性。

In [ ]:
# 每个行为级别的用户数
funnel = pd.DataFrame({
    'stage': ['浏览(pv)', '收藏(fav)', '加购(cart)', '购买(buy)'],
    'users': [
        df[df['behavior_type']=='pv']['user_id'].nunique(),
        df[df['behavior_type']=='fav']['user_id'].nunique(),
        df[df['behavior_type']=='cart']['user_id'].nunique(),
        df[df['behavior_type']=='buy']['user_id'].nunique(),
    ]
})
funnel['prev_users'] = funnel['users'].shift(1)
funnel['step_rate'] = (funnel['users'] / funnel['prev_users'] * 100).round(1)
funnel['overall_rate'] = (funnel['users'] / funnel['users'].iloc[0] * 100).round(1)
print(funnel)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 漏斗图
ax = axes[0]
widths = funnel['users'].values / funnel['users'].max()
colors = ['#4C72B0', '#55A868', '#DD8452', '#C44E52']
for i in range(len(funnel)):
    ax.barh(i, widths[i], color=colors[i], height=0.6,
            label=funnel['stage'].iloc[i])
    ax.text(widths[i] + 0.02, i,
            f'{funnel["users"].iloc[i]:,}人 ({funnel["overall_rate"].iloc[i]}%)',
            va='center', fontweight='bold')
ax.set_yticks(range(len(funnel)))
ax.set_yticklabels(funnel['stage'])
ax.set_xlim(0, 1.2)
ax.set_title('整体转化漏斗（用户数）')
ax.invert_yaxis()

# 每步流失率
ax = axes[1]
stages_connect = ['浏览→收藏', '收藏→加购', '加购→购买']
rates = [funnel['step_rate'].iloc[i] for i in range(1, 4)]
bars = ax.bar(stages_connect, rates, color=['#55A868', '#DD8452', '#C44E52'])
ax.axhline(y=rates[0], color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('转化率 (%)')
ax.set_title('每步转化率')
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{rate}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/fig_funnel.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. 分品类漏斗对比

不同品类的转化行为可能天差地别——高频低客单价 vs 低频高客单价，漏斗形状完全不同。

In [ ]:
# 用 SQL 做分品类漏斗（窗口函数 + CASE WHEN）
category_funnel_sql = """
WITH cat_behavior AS (
    SELECT category_id, behavior_type, COUNT(DISTINCT user_id) AS uv
    FROM user_behavior_clean
    WHERE date < '2017-12-08'
    GROUP BY category_id, behavior_type
)
SELECT 
    category_id,
    MAX(CASE WHEN behavior_type='pv' THEN uv END) AS pv_users,
    MAX(CASE WHEN behavior_type='fav' THEN uv END) AS fav_users,
    MAX(CASE WHEN behavior_type='cart' THEN uv END) AS cart_users,
    MAX(CASE WHEN behavior_type='buy' THEN uv END) AS buy_users
FROM cat_behavior
GROUP BY category_id
"""
cat_funnel = query(category_funnel_sql)
cat_funnel['purchase_rate'] = (cat_funnel['buy_users'] / cat_funnel['pv_users'] * 100).round(2)
cat_funnel = cat_funnel.sort_values('purchase_rate', ascending=False)
cat_funnel = cat_funnel[cat_funnel['pv_users'] > 50]  # 过滤样本太少的品类
print(f'有效品类数: {len(cat_funnel)}')
print(f'\n转化率最高 Top 5 品类:')
print(cat_funnel.head(5)[['category_id', 'pv_users', 'buy_users', 'purchase_rate']])
print(f'\n转化率最低 Bottom 5 品类:')
print(cat_funnel.tail(5)[['category_id', 'pv_users', 'buy_users', 'purchase_rate']])
print(f'\n品类转化率分布: 均值={cat_funnel["purchase_rate"].mean():.1f}%, 中位数={cat_funnel["purchase_rate"].median():.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(cat_funnel['purchase_rate'], bins=30, color='#4C72B0', alpha=0.7, edgecolor='white')
ax.axvline(cat_funnel['purchase_rate'].mean(), color='red', linestyle='--',
           label=f'均值={cat_funnel["purchase_rate"].mean():.1f}%')
ax.axvline(cat_funnel['purchase_rate'].median(), color='green', linestyle='--',
           label=f'中位数={cat_funnel["purchase_rate"].median():.1f}%')
ax.set_xlabel('品类购买转化率 (%)')
ax.set_ylabel('品类数')
ax.set_title('各品类 PV→购买 转化率分布')
ax.legend()
plt.tight_layout()
plt.savefig('../data/fig_category_funnel.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 用户路径分析

用户实际走了哪些路径？不是所有用户都走 pv→fav→cart→buy 标准流程。

In [ ]:
# 每个用户在每个商品上的行为聚合为路径
path_sql = """
SELECT 
    user_id, item_id,
    GROUP_CONCAT(DISTINCT behavior_type ORDER BY time) AS path
FROM user_behavior_clean
WHERE date < '2017-12-08'
GROUP BY user_id, item_id
"""
paths = query(path_sql)
print(f'总共 {len(paths):,} 个用户-商品交互对')

# 统计 Top 路径
path_counts = paths['path'].value_counts().head(10)
path_pct = (path_counts / len(paths) * 100).round(2)
print(f'\nTop 10 用户路径:')
print(pd.DataFrame({'次数': path_counts, '占比(%)': path_pct}))

In [ ]:
# 定义关键路径类型
def classify_path(p):
    if p == 'pv': return '仅浏览'
    if 'buy' in p and 'cart' in p: return '加购后购买'
    if 'buy' in p and 'fav' in p and 'cart' not in p: return '收藏后购买'
    if 'buy' in p and 'fav' not in p and 'cart' not in p: return '直接购买(浏览→买)'
    if 'cart' in p and 'buy' not in p: return '加购未买'
    if 'fav' in p and 'cart' not in p and 'buy' not in p: return '收藏未买'
    return '其他'

paths['path_type'] = paths['path'].apply(classify_path)
path_type_counts = paths['path_type'].value_counts()
print('用户路径类型分布:')
print(path_type_counts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_path = ['#4C72B0', '#55A868', '#DD8452', '#C44E52', '#8E44AD', '#E67E22', '#1ABC9C']
axes[0].pie(path_type_counts.values, labels=path_type_counts.index,
            autopct='%1.1f%%', colors=colors_path, startangle=90)
axes[0].set_title('用户路径类型')

# 有购买的路径中，加购 vs 直接买 vs 收藏
buy_paths = paths[paths['path'].str.contains('buy')]
buy_types = buy_paths['path_type'].value_counts()
axes[1].bar(range(len(buy_types)), buy_types.values, color=['#C44E52', '#DD8452', '#8E44AD'])
axes[1].set_xticks(range(len(buy_types)))
axes[1].set_xticklabels(buy_types.index, rotation=15)
axes[1].set_title('购买用户的路径拆解')
for i, v in enumerate(buy_types.values):
    axes[1].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/fig_paths.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 关键发现

In [ ]:
pv_only = paths[paths['path'] == 'pv'].shape[0]
cart_no_buy = paths[paths['path_type'] == '加购未买'].shape[0]
print('=== 转化漏斗关键发现 ===')
print(f'1. 浏览次数 → 购买: 整体转化率约为 funnel["overall_rate"].iloc[-1]%')
print(f'2. 最大流失点: {path_type_counts.index[0]}（{path_type_counts.values[0]:,} 对）, 占比 {path_type_counts.values[0]/len(paths)*100:.1f}%')
print(f'3. 品类间转化率差异大: 最高 {cat_funnel["purchase_rate"].max():.1f}% vs 最低 {cat_funnel["purchase_rate"].min():.1f}%')
print(f'4. 加购但未买的用户-商品对有 {cart_no_buy:,} 对——这是最大的可挽回空间')
print(f'5. 收藏后再买的路径占比很小，说明收藏功能可能主要用作"标记"而非购买前奏')